In [15]:
# Limite les threads BLAS au niveau global aussi (utile même côté parent)
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import sys
from pathlib import Path
SRC = (Path.cwd().parent / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
import seaborn as sns
import utils.features_extraction as features_extraction
import utils.metrics as metrics
import utils.optimization as optimization


import warnings
import importlib, params

importlib.reload(params)
importlib.reload(features_extraction)
importlib.reload(metrics)
importlib.reload(optimization)


warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid")

In [16]:
# Load data from SAS pickle file
data = pd.read_parquet(os.path.join(params.DATA_FOLDER, 'wrds\sp500_daily_full.parquet'), 
                       columns=['date', 'permno', 'ret_combined', 'me', 'dollar_vol'])

In [17]:
data.head()

,date,permno,ret_combined,me,dollar_vol
0,2000-01-03,10078,-0.012107,1.194246e+08,1.168195e+09
1,2000-01-03,10104,0.054099,1.681713e+08,2.933259e+09
2,2000-01-03,10107,-0.001606,6.014654e+08,3.139858e+09
3,2000-01-03,10138,-0.049069,4.238815e+06,1.340426e+07
4,2000-01-03,10145,-0.017335,4.473965e+07,1.190890e+08


In [10]:
data_pivot= data[(data['date'] >= params.START_DATE) & (data['date'] <= params.END_DATE)].pivot(index='date', 
                                                                                         columns='permno', 
                                                                                         values=['ret_combined', 'me', 'dollar_vol'])
# Extract unique PERMNOs from SAS
permnos = data_pivot["ret_combined"].columns.tolist()
permnos = [permno for permno in permnos if permno not in (np.nan, None) and permno != '']

print(f"Working with {len(permnos)} PERMNOs and a total of {data_pivot["ret_combined"].shape[0]} days over the period {params.START_DATE} to {params.END_DATE}.")

df_return = data_pivot["ret_combined"]
df_return.head()

Working with 520 PERMNOs and a total of 252 days over the period 2024-01-01 to 2024-12-31.


permno,10104,10107,10138,10145,10516,10696,11308,11403,11404,11533,...,92611,92614,92655,93002,93089,93096,93132,93246,93429,93436
date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,-0.012994,-0.013749,0.002043,-0.003386,0.007477,0.001807,0.015103,-0.034769,0.015829,-0.028221,...,-0.005685,0.011516,0.024446,-0.027655,-0.003684,0.032953,-0.012814,-0.014469,-0.002016,-0.000241
2024-01-03,-0.015376,-0.000728,-0.015013,-0.021388,0.005085,-0.009994,0.002340,-0.021377,0.006385,-0.007771,...,-0.003438,0.000000,0.004988,-0.024692,-0.005379,-0.065157,0.001211,-0.060061,-0.018631,-0.040134
2024-01-04,0.001269,-0.007178,-0.002164,0.001809,-0.018871,0.009488,-0.003336,-0.005286,0.002688,-0.004375,...,0.006318,-0.013108,0.006254,-0.009040,0.005281,0.026508,0.009334,-0.028316,-0.007605,-0.002181
2024-01-05,0.001365,-0.000516,0.001037,-0.006686,-0.013101,-0.003233,-0.001506,-0.012543,0.002895,-0.001790,...,0.000308,0.000987,-0.014741,0.000257,-0.017861,0.007421,-0.000856,0.011089,-0.002420,-0.001849
2024-01-08,0.018787,0.018872,0.007158,-0.004275,0.003531,0.020065,0.007374,0.036405,0.003101,0.033143,...,0.010975,0.008719,-0.001600,0.024369,-0.004236,-0.006998,0.050051,0.028567,0.018888,0.012464


#### Market Capitalization

In [18]:
rebalance_idx = features_extraction.generate_rebalance_dates(df_return.index, frequency=params.REBALANCING_FREQUENCY)
rebalance_indices = [df_return.index.get_loc(d) for d in rebalance_idx]